# Campus Threads · Part 2
**Данные синтетические.** 600 вымышленных аккаунтов, 18 сообществ и 32 мероприятия. Мы моделируем работу приложения и проверяем, как дашборд отвечает на вопросы о регистрации, активности и пропусках.

Нажми **Runtime → Run all** и загрузи `campus_threads_dataset.zip`, когда первая ячейка попросит файл. Внизу появятся пять графиков. Каждый блок ниже показывает, из каких столбцов считается метрика.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

archive = Path('campus_threads_dataset.zip')
if not archive.exists():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Выбери один файл campus_threads_dataset.zip.')
    archive = Path(next(iter(uploaded)))

with ZipFile(archive) as bundle:
    info = json.loads(bundle.read('dataset_info.json'))
    def read_csv(name, dates=()):
        with bundle.open(name) as stream:
            return pd.read_csv(stream, encoding='utf-8-sig', parse_dates=list(dates))

    activity = read_csv('bi/activity.csv', ['date', 'occurred_at'])
    users = read_csv('tables/users.csv', ['registered_at'])
    faculties = read_csv('tables/faculties.csv')
    communities = read_csv('tables/communities.csv')
    members = read_csv('tables/community_members.csv', ['joined_at'])
    events = read_csv('bi/events.csv', ['starts_at'])

if 'google.colab' in __import__('sys').modules:
    pio.renderers.default = 'colab'
palette = ['#3148ab', '#009489', '#d97717', '#7752b6', '#c14665', '#6a86a0', '#79a440', '#bd7233', '#8b94a8']
config = {'displaylogo': False, 'responsive': True}

def styled(fig, height=360):
    fig.update_layout(template='plotly_white', autosize=True, height=height,
                      margin=dict(l=48, r=28, t=21, b=45),
                      font=dict(family='Arial, sans-serif', size=13, color='#172849'),
                      paper_bgcolor='white', plot_bgcolor='white',
                      colorway=palette, legend_title_text='')
    fig.update_xaxes(automargin=True, gridcolor='#edf0f6')
    fig.update_yaxes(automargin=True, gridcolor='#edf0f6')
    return fig

def select_period(frame, column, start, end):
    return frame[frame[column].ge(start) & frame[column].lt(end + pd.Timedelta(days=1))]

def blank(message='В выбранном диапазоне нет данных'):
    fig = go.Figure()
    fig.add_annotation(text=message, x=.5, y=.5, showarrow=False, xref='paper', yref='paper')
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return styled(fig)

print(f"Синтетические данные | {info['start_date']} — {info['end_date']}")

## 1. Линии · новые и активные пользователи

In [ ]:
def weekly_chart(rows, start, end):
    weeks = pd.date_range(start.to_period('W-SUN').start_time, end.to_period('W-SUN').start_time, freq='W-MON')
    signed = rows.loc[rows.action.eq('register')].groupby(pd.Grouper(key='date', freq='W-MON', label='left', closed='left')).size()
    active = rows.loc[rows.action.ne('register')].groupby(pd.Grouper(key='date', freq='W-MON', label='left', closed='left')).user_id.nunique()
    fig = go.Figure()
    for name, series, color in [('Регистрации', signed, palette[0]), ('Активные пользователи', active, palette[1])]:
        fig.add_scatter(x=weeks, y=series.reindex(weeks, fill_value=0), mode='lines+markers',
                        name=name, line=dict(color=color, width=3), hovertemplate='%{x|%d.%m}<br>%{y} чел.')
    fig.update_layout(xaxis_title='Неделя с понедельника', yaxis_title='Человек', legend=dict(orientation='h', y=1.12))
    fig.update_xaxes(tickformat='%d.%m', tick0=weeks[0], dtick=14*86400000)
    fig.update_yaxes(rangemode='tozero')
    return styled(fig)

## 2. Горизонтальные столбцы · размер сообществ

In [ ]:
def clubs_chart(end, user_ids):
    # Действующие членства из снимка, возникшие к выбранной дате окончания.
    selected = members[members.user_id.isin(user_ids) & members.joined_at.lt(end + pd.Timedelta(days=1))]
    ranking = (selected.groupby('community_id').size().rename('members').reset_index()
               .merge(communities[['community_id', 'name']], on='community_id', validate='one_to_one')
               .sort_values(['members', 'name'], ascending=[False, True]).head(8).iloc[::-1])
    if ranking.empty:
        return blank()
    fig = px.bar(ranking, x='members', y='name', orientation='h', text='members',
                 labels={'members': 'Участников', 'name': ''}, color_discrete_sequence=[palette[0]])
    fig.update_layout(yaxis_title=None)
    fig.update_traces(textposition='outside', cliponaxis=False)
    fig.update_xaxes(range=[0, ranking.members.max()*1.22])
    return styled(fig)

## 3. Точки · заполнение мест на мероприятиях

In [ ]:
def events_chart(start, end, category):
    selected = select_period(events, 'starts_at', start, end)
    if category != 'Все':
        selected = selected[selected.category.eq(category)]
    if selected.empty:
        return blank('В выбранном диапазоне нет мероприятий')
    fig = px.scatter(selected, x='capacity', y='fill_pct', color='category', hover_name='title',
                     hover_data={'tickets_issued': True, 'starts_at': '|%d.%m.%Y', 'capacity': True, 'fill_pct': ':.1f'},
                     labels={'capacity': 'Вместимость, мест', 'fill_pct': 'Забронировано, %',
                             'category': 'Категория', 'tickets_issued': 'Выдано пропусков', 'starts_at': 'Начало'},
                     color_discrete_sequence=palette)
    fig.update_traces(marker=dict(size=12, opacity=.85, line=dict(color='white', width=1)))
    fig.add_hline(y=100, line_dash='dot', line_color='#93a2bb')
    fig.update_yaxes(range=[0, 107], ticksuffix='%')
    fig.update_layout(legend=dict(orientation='h', y=-.25))
    return styled(fig, 400)

## 4. Тепловая карта · день недели и час

In [ ]:
def heatmap_chart(rows):
    labels = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
    grid = rows.groupby(['weekday', 'hour']).size().unstack(fill_value=0).reindex(index=range(1, 8), columns=range(24), fill_value=0)
    fig = go.Figure(go.Heatmap(x=list(range(24)), y=labels, z=grid.to_numpy(),
        colorscale=[[0, '#edf2ff'], [.5, '#6e8cec'], [1, '#223b97']], zmin=0,
        hovertemplate='%{y}, %{x}:00<br>%{z} действий<extra></extra>', colorbar=dict(title='Действий')))
    fig.update_layout(xaxis_title='Час по Алматы', yaxis_autorange='reversed')
    fig.update_xaxes(dtick=3)
    return styled(fig)

## 5. Кольцевая диаграмма · действия по модулям

In [ ]:
def usage_chart(rows):
    metrics = rows.groupby('action_group').size().sort_values(ascending=False)
    if metrics.empty:
        return blank()
    texts = [f'{value/metrics.sum():.1%}' if value/metrics.sum() >= .035 else '' for value in metrics]
    fig = go.Figure(go.Pie(labels=metrics.index, values=metrics.values, hole=.58,
                           text=texts, textinfo='text', textposition='inside', marker=dict(colors=palette),
                           hovertemplate='%{label}<br>%{value} действий (%{percent})<extra></extra>'))
    fig.update_layout(legend=dict(orientation='h', y=-.02), margin=dict(b=85))
    fig = styled(fig, 405)
    fig.update_layout(margin=dict(l=20, r=20, t=18, b=87))
    return fig

## Дашборд и фильтры

Измени параметры в последней ячейке. При необходимости запусти её ещё раз кнопкой ▶. Факультет фильтрует активность и членство. Категория применяется только к графику мероприятий.

In [ ]:
chart_text = [
    ('Регистрации и активные пользователи', 'Активные совершили хотя бы одно действие помимо регистрации. Крайние недели могут быть неполными.'),
    ('Крупнейшие сообщества', 'Текущие членства из итогового снимка, добавленные до даты окончания. Отбор по факультету.'),
    ('Заполняемость мероприятий', 'Выданные пропуска / вместимость. Учитываются все факультеты. Фильтры по дате события и категории.'),
    ('Активность по дням и часам', 'Действия пользователей по местному времени. Симулированные просмотры и поиск включены.'),
    ('Использование функций', 'Доля действий пользователей по модулям. Операции администратора базы исключены.'),
]

def make_dashboard(faculty='Все', date_from=info['start_date'], date_to=info['end_date'], category='Все'):
    start, end = pd.Timestamp(date_from).normalize(), pd.Timestamp(date_to).normalize()
    if pd.isna(start) or pd.isna(end) or not pd.Timestamp(info['start_date']) <= start <= end <= pd.Timestamp(info['end_date']):
        raise ValueError('Выбери даты внутри периода 01.09–23.11.2026. Начало должно быть не позже конца.')
    user_ids = users.user_id
    rows = activity[activity.user_id.notna()]
    if faculty != 'Все':
        ids = faculties.loc[faculties.faculty_code.eq(faculty), 'faculty_id']
        user_ids = users.loc[users.faculty_id.isin(ids), 'user_id']
        rows = rows[rows.faculty_code.eq(faculty)]
    rows = select_period(rows, 'date', start, end)
    kpis = [int(rows.action.eq('register').sum()), int(rows.loc[rows.action.ne('register'), 'user_id'].nunique()),
            int(rows.action.eq('ticket_claim').sum()), len(rows)]
    return kpis, [weekly_chart(rows, start, end), clubs_chart(end, user_ids),
                  events_chart(start, end, category), heatmap_chart(rows), usage_chart(rows)]

def show_dashboard(faculty='Все', date_from=info['start_date'], date_to=info['end_date'], category='Все'):
    from IPython.display import Markdown, display
    kpis, figures = make_dashboard(faculty, date_from, date_to, category)
    display(Markdown(f"**Синтетические данные** · {date_from} — {date_to} · Факультет: {faculty}\n\n"
                     f"**Регистрации:** {kpis[0]:,} · **Активные:** {kpis[1]:,} · **Пропуска:** {kpis[2]:,} · **Действия:** {kpis[3]:,}"))
    for (title, note), figure in zip(chart_text, figures):
        display(Markdown(f'### {title}\n{note}'))
        figure.show(config=config)

In [ ]:
#@title Фильтры дашборда { run: "auto" }
Факультет = "Все" #@param ["Все", "SITE", "BS", "SAM", "SEOGI", "ISE", "SCE"]
Начало = "2026-09-01" #@param {type:"date"}
Конец = "2026-11-23" #@param {type:"date"}
Категория_мероприятий = "Все" #@param ["Все", "Волонтёрство", "Кампус", "Карьера", "Клубы", "Спорт", "Учёба"]

try:
    show_dashboard(Факультет, Начало, Конец, Категория_мероприятий)
except ValueError as error:
    print(error)